# Week 54, Day 1: What it means to learn from data

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w054/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Minimising the training error over a class of lines

Write emp_risk(a, b), the mean squared error of the line ax + b on the training data. Evaluate it on the grid of slopes and intercepts, find the minimiser, and compare it with the least-squares solution from np.linalg.lstsq.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(54)
f = lambda x: 0.8 * np.sin(np.pi * x)
n, sigma = 20, 0.3
x = r.uniform(-1, 1, n); y = f(x) + sigma * r.normal(size=n)

def emp_risk(a, b):
    return ...

slopes, intercepts = np.arange(-2, 2.001, 0.02), np.arange(-1, 1.001, 0.02)
R = np.array([[emp_risk(a, b) for b in intercepts] for a in slopes])
i, j = np.unravel_index(np.argmin(R), R.shape)
a_grid, b_grid = slopes[i], intercepts[j]
(a_ls, b_ls), *_ = np.linalg.lstsq(np.c_[x, np.ones(n)], y, rcond=None)
print(f"grid search: a = {a_grid:.2f}, b = {b_grid:.2f}, training risk {R[i, j]:.4f}")
print(f"least squares: a = {a_ls:.3f}, b = {b_ls:.3f}, training risk {emp_risk(a_ls, b_ls):.4f}")
plt.contourf(intercepts, slopes, R, 30); plt.plot(b_ls, a_ls, "w*"); plt.xlabel("intercept b"); plt.ylabel("slope a"); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_risk():
    "emp_risk(a, b) is the mean squared error of the line on the training data"
    assert abs(emp_risk(0.3, -0.1) - np.mean((y - (0.3 * x - 0.1)) ** 2)) < 1e-12
def test_grid():
    "R holds the empirical risk on the grid and (a_grid, b_grid) is its minimiser"
    i, j = np.unravel_index(np.argmin(R), R.shape)
    assert R.shape == (len(slopes), len(intercepts)) and a_grid == slopes[i] and b_grid == intercepts[j]
def test_ls():
    "the grid minimiser is within one grid step of the least-squares line"
    assert abs(a_grid - a_ls) <= 0.02 and abs(b_grid - b_ls) <= 0.02

test_risk()
test_grid()
test_ls()
print("All checks passed")

### Exercise: Training error flatters, by how much?

For each degree (1 and 9) and training size (20 and 200), draw 200 training sets, fit a polynomial by least squares (np.polyfit), and record its training risk and its expected risk, estimated on a fresh sample of 5,000 points. Store the medians over training sets in results (a few fits explode, so the mean is misleading).

In [ ]:
x_big = r.uniform(-1, 1, 5000); y_big = f(x_big) + sigma * r.normal(size=5000)
results = {}
for deg in [1, 9]:
    for n_tr in [20, 200]:
        tr, te = [], []
        for rep in range(200):
            xt = r.uniform(-1, 1, n_tr); yt = f(xt) + sigma * r.normal(size=n_tr)
            w = np.polyfit(xt, yt, deg)
            tr.append(...)
            te.append(...)
        results[deg, n_tr] = (np.median(tr), np.median(te))
        print(f"degree {deg}, n = {n_tr:3d}: median training risk {np.median(tr):.4f}, median expected risk {np.median(te):.4f}, worst expected risk {np.max(te):.1f}")

In [ ]:
import numpy as np
def test_shape():
    "results maps (degree, n) to the median training and expected risk over 200 training sets"
    assert set(results) == {(1, 20), (9, 20), (1, 200), (9, 200)} and all(len(v) == 2 for v in results.values())
def test_gap():
    "the training risk is below the expected risk for every case"
    assert all(tr < te for tr, te in results.values())
def test_pattern():
    "the gap is largest for the flexible model with little data, and shrinks with more data"
    gap = {k: te - tr for k, (tr, te) in results.items()}
    assert gap[(9, 20)] > gap[(1, 20)] and gap[(9, 200)] < gap[(9, 20)] / 5

test_shape()
test_gap()
test_pattern()
print("All checks passed")

### Exercise: Mean, median and skewed spike counts

Spike counts from a bursty neuron are skewed. For each constant prediction c on the grid, compute the mean squared loss and the mean absolute loss over the counts, find each minimiser, and compare with the mean and median.

In [ ]:
counts = r.negative_binomial(2, 0.25, 2000)          # skewed, overdispersed counts
cs = np.arange(0, 15.001, 0.01)
sq = ...
ab = ...
c_sq, c_abs = cs[np.argmin(sq)], cs[np.argmin(ab)]
print(f"squared loss minimised at {c_sq:.2f}; sample mean {counts.mean():.3f}")
print(f"absolute loss minimised at {c_abs:.2f}; sample median {np.median(counts):.1f}")

In [ ]:
import numpy as np
def test_minimisers():
    "c_sq and c_abs minimise the squared and absolute losses on the grid"
    assert c_sq == cs[np.argmin(sq)] and c_abs == cs[np.argmin(ab)]
def test_match():
    "they match the sample mean and median within one grid step"
    assert abs(c_sq - counts.mean()) <= 0.01 and abs(c_abs - np.median(counts)) <= 0.5

test_minimisers()
test_match()
print("All checks passed")